In [4]:
# pdf에서 페이지 정보 json으로 변환하는 프로그램
import os
import json
import numpy as np
from PIL import Image
from pdf2image import convert_from_path
from paddleocr import PaddleOCR
import re





# pdf to images 함수
def convert_pdf_to_images(pdf_path, dpi=300):
    images = convert_from_path(pdf_path, dpi=dpi)
    output_dir = os.path.splitext(pdf_path)[0]
    os.makedirs(output_dir, exist_ok=True)
    image_paths = []

    
    for i, image in enumerate(images):
        
        img_path = os.path.join(output_dir, f"{os.path.basename(output_dir)}_{i}.png")
        
        image.save(img_path, "PNG")
        
        image_paths.append(img_path)

    return image_paths






# json 변환 함수
def convert_for_json(obj):
    if isinstance(obj, (list, tuple)):
        return [convert_for_json(o) for o in obj]
    elif isinstance(obj, dict):
        return {k: convert_for_json(v) for k, v in obj.items()}
    elif isinstance(obj, (int, float, str)):
        return obj
    else:
        return str(obj)




# 이미지 하단 일정 높이만 추출
def crop_bottom_region(image, height):
    x1 = 0
    x2 = image.width
    y2 = image.height
    y1 = y2 - height
    
    if y1 < 0:
        raise ValueError("입력한 높이가 이미지보다 큽니다")

    crop_box = (x1, y1, x2, y2)
    cropped = image.crop(crop_box)
    return np.array(cropped), crop_box




# 문자열 좌표 추출
def extract_coords(poly_str):
    coords = re.findall(r"\[(\d+)\s+(\d+)\]", poly_str)
    return [(int(x), int(y)) for x, y in coords]




# 좌표 평균 x값
def get_center_x(coords):
    xs = [pt[0] for pt in coords]
    return np.mean(xs)




# left right 위치 부여
def add_position_to_ocr_result(ocr_result, image_width):
    center_x = image_width / 2
    
    updated_result = []

    for res in ocr_result:
        dt_polys = res.get("dt_polys", [])
        rec_texts = res.get("rec_texts", [])
        positions = []

        
        for poly_str in dt_polys:
            
            if not poly_str or poly_str == "[]":
                positions.append("unknown")
                continue
                
            coords = extract_coords(poly_str)
            cx = get_center_x(coords)
            pos = "left" if cx < center_x else "right"
            positions.append(pos)

        updated_res = []
        for idx, text in enumerate(rec_texts):
            position = positions[idx] if idx < len(positions) else "unknown"
            coords = extract_coords(dt_polys[idx]) if idx < len(dt_polys) else []
            updated_res.append({
                "text": text,
                "position": position,
                "coords": coords
            })

        new_res = dict(res)
        new_res["rec_texts_with_position"] = updated_res
        updated_result.append(new_res)

    return updated_result




# 숫자 텍스트 여부 확인 0 ~ 9999 범위
def is_valid_number(text):
    if not text.strip().isdigit():
        return False
    num = int(text.strip())
    return 0 <= num <= 9999



    
# 이미지 파일 넘버 정렬
def natural_keys(text):
    return [int(s) if s.isdigit() else s.lower() for s in re.split(r'(\d+)', text)]


    

# main entry
if __name__ == "__main__":
    pdf_path = input("pdf 경로를 입력: ").strip()
    image_paths = convert_pdf_to_images(pdf_path)
    print(f"\n{len(image_paths)}장 이미지 변환 완료\n")

    
    image_dir = os.path.dirname(image_paths[0])
    height = int(input("인식할 픽셀 높이를 입력: ").strip())
    ocr = PaddleOCR(lang='korean')

    
    valid_exts = [".png", ".jpg", ".jpeg"]
    image_files = [f for f in os.listdir(image_dir) if os.path.splitext(f)[1].lower() in valid_exts]
    
    if not image_files:
        print("해당 디렉토리에 이미지가 존재하지 않음")
        exit(1)

    # 이미지 파일명 정렬
    image_files.sort(key=natural_keys)
    all_results = []

    
    for filename in image_files:
        image_path = os.path.join(image_dir, filename)
        
        try:
            image = Image.open(image_path)
            cropped_np, crop_box = crop_bottom_region(image, height)
            image_width = crop_box[2] - crop_box[0]

            result = ocr.predict(cropped_np)
            converted_result = convert_for_json(result)

            
            # 페이지숫자 ocr인식시 0 ~ 9999 사이만 인식
            for res in converted_result:
                texts = res.get("rec_texts", [])
                filtered = [t for t in texts if is_valid_number(t)]
                res["rec_texts"] = filtered

            result_with_pos = add_position_to_ocr_result(converted_result, image_width)

            
            empty_text_flag = True

            
            for res in converted_result:
                rec_texts = res.get("rec_texts", [])
                if any(text.strip() for text in rec_texts):
                    empty_text_flag = False
                    break

            
            result_json = {
                "filename": filename,
                "ocr_result": result_with_pos,
                "empty_text": empty_text_flag
            }

            
            all_results.append(result_json)
            print(f"OCR 인식 완료: {filename}")

        
        except Exception as e:
            print(f"{filename} 오류발생: {e}")


    
    output_path = os.path.join(image_dir, "pdf_page_ocr_results.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)


    
    print(f"\n결과 json으로 저장 완료: {output_path}")


pdf 경로를 입력:  /home/zen35/Desktop/pdfwork/imgdata/t2/6/2.pdf



70장 이미지 변환 완료



인식할 픽셀 높이를 입력:  500


/home/zen35/anaconda3/envs/yolowork/lib/python3.10/site-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in /home/zen35/.paddlex/official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

...

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('korean_PP-OCRv5_mobile_rec', None)
Using official model (korean_PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in /home/zen35/.paddlex/official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

OCR 인식 완료: 2_0.png
OCR 인식 완료: 2_1.png
OCR 인식 완료: 2_2.png


KeyboardInterrupt: 

In [3]:
# 위의 코드에서 pdf의 페이지 수 정보를 json변환한후 나온 json을 이 코드에서 분석

import json
import os

def analyze_pdf_page_patterns():
    json_path = input("json 결과 경로 입력: ").strip()

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    
    # 첫 페이지부터 끝까지 left right 검사 
    alternating_results = []
    base_position = None
    expected = None
    base_filename = None
    found_base = False

    alternating_wrong_files = set()

    
    for item in data:
        filename = item["filename"]
        
        empty = item.get("empty_text", True)

        if not found_base and not empty:
            rec_infos = item.get("ocr_result", [])
            
            positions = []
            
            for block in rec_infos:
                for res in block.get("rec_texts_with_position", []):
                    
                    pos = res.get("position", "unknown")
                    
                    if pos in ["left", "right"]:
                        positions.append(pos)

            if not positions:
                alternating_results.append((filename, "no position", "base but no position"))
                continue

            left_count = positions.count("left")
            right_count = positions.count("right")
            base_position = "left" if left_count >= right_count else "right"
            expected = "left" if base_position == "right" else "right"
            base_filename = filename
            found_base = True

            alternating_results.append((filename, base_position, "base"))
            continue

        
        if not found_base:
            alternating_results.append((filename, "empty", "base 페이지 이전까지 스킵"))
            continue

        current_position = None

        if not empty:
            rec_infos = item.get("ocr_result", [])
            
            positions = []
            
            for block in rec_infos:
                
                for res in block.get("rec_texts_with_position", []):
                    pos = res.get("position", "unknown")
                    
                    if pos in ["left", "right"]:
                        positions.append(pos)

            if not positions:
                alternating_results.append((filename, "no position", f"expected {expected}"))
                expected = "right" if expected == "left" else "left"
                continue

            left_count = positions.count("left")
            right_count = positions.count("right")
            current_position = "left" if left_count >= right_count else "right"

            if current_position == expected:
                alternating_results.append((filename, current_position, "OK"))
            else:
                alternating_results.append((filename, current_position, f"WRONG (expected {expected})"))
                alternating_wrong_files.add(filename)
                
        else:
            alternating_results.append((filename, "empty", f"넘기는 페이지, 예상 패턴 {expected}"))
        
        expected = "right" if expected == "left" else "left"





    
    # 생략되거나(건너뛴) 중복된 페이지 숫자 확인
    number_sequence = []

    for item in data:
        
        filename = item["filename"]
        
        ocr_result = item.get("ocr_result", [])
        
        if not ocr_result:
            continue
        
        texts_with_pos = ocr_result[0].get("rec_texts_with_position", [])

        
        for obj in texts_with_pos:
            
            text = obj.get("text", "")
            
            if text.isdigit():
                number_sequence.append((int(text), filename))

    number_sequence.sort()
   
    skipped_files = []
    
   
    for i in range(1, len(number_sequence)):
        
        current_num, current_file = number_sequence[i]
        
        prev_num, _ = number_sequence[i - 1]
        
        if current_num != prev_num + 1:
            skipped_files.append(current_file)





    
    # 중복된 left right 방향 탐지
    prev_position = None
    
    violations = []
    
    for item in data:
        
        filename = item["filename"]
        
        ocr_result = item.get("ocr_result", [])
        
        if not ocr_result:
            continue
        texts_with_pos = ocr_result[0].get("rec_texts_with_position", [])
        
        if not texts_with_pos:
            continue
            
        current_position = texts_with_pos[0].get("position", "").lower()
        
        if prev_position is not None and current_position == prev_position:
            violations.append({
                "filename": filename,
                "position": current_position
            })
            
        prev_position = current_position




    

    
    # 탐지된 파일 저장
    skipped_set = set(skipped_files)
    violation_set = set(v["filename"] for v in violations)
    alternating_set = set(alternating_wrong_files)
    problem_set = skipped_set | violation_set | alternating_set

    result_data = []
    
    for item in data:
        filename = item["filename"]
        
        if filename not in problem_set:
            continue

        ocr_result = item.get("ocr_result", [])
        
        if not ocr_result:
            continue
            
        texts_with_pos = ocr_result[0].get("rec_texts_with_position", [])

        code = []
        if filename in skipped_set:
            code.append(1)
            
        if filename in violation_set:
            code.append(2)
            
        if filename in alternating_set:
            code.append(3)

        error_code = code[0] if len(code) == 1 else code

        output = {
            "filename": filename,
            "rec_texts_with_position": texts_with_pos,
            "error_code": error_code
        }

        result_data.append(output)



    
    # 모든 탐지 상황 저장
    all_info = {
        "base_filename": base_filename,
        "base_position": base_position,
        "left_right_pattern": [
            {"filename": fname, "position": pos, "status": status}
            for fname, pos, status in alternating_results
        ],
        "skipped_files": skipped_files,
        "violations": violations
    }


    
    # 탐지 결과 json 저장
    output_dir = os.path.dirname(json_path)
    output_path = os.path.join(output_dir, "pdf_result.json")

    final_output = {
        "error_results": result_data,
        "all_info": all_info
    }

    
    # json 덤핑
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_output, f, ensure_ascii=False, indent=2)



    

    
    # 콘솔 출력
    print("\n탐지된 비정상 파일")
    for entry in result_data:
        print(json.dumps(entry, ensure_ascii=False, indent=2))

    
    print("\nleft right 탐지 정보")
    print(f"기준 페이지: {base_filename} → '{base_position}' 시작")
    print("\n")
    for fname, pos, status in alternating_results:
        print(f"{fname:20} | {pos:8} | {status}")

    
    print("\n건너뛴 숫자가 있는 파일들")
    for fname in skipped_files:
        print(f"  - {fname}")

    
    print("\n중복된 left, right")
    for v in violations:
        print(f"  - {v['filename']} 에서 '{v['position']}' 중복 발생")

    
    print(f"\n탐지 결과가 '{output_path}'에 저장")



# main entry
if __name__ == "__main__":
    analyze_pdf_page_patterns()


json 결과 경로 입력:  /home/zen35/Desktop/pdfwork/imgdata/t2/6/2/pdf_page_ocr_results.json



탐지된 비정상 파일
{
  "filename": "2_44.png",
  "rec_texts_with_position": [
    {
      "text": "00711",
      "position": "right",
      "coords": [
        [
          2163,
          0
        ],
        [
          2163,
          13
        ]
      ]
    },
    {
      "text": "37",
      "position": "right",
      "coords": [
        [
          2316,
          114
        ],
        [
          2316,
          159
        ]
      ]
    }
  ],
  "error_code": 1
}
{
  "filename": "2_54.png",
  "rec_texts_with_position": [
    {
      "text": "00711",
      "position": "right",
      "coords": [
        [
          2157,
          0
        ],
        [
          2157,
          13
        ]
      ]
    },
    {
      "text": "47",
      "position": "right",
      "coords": [
        [
          2326,
          117
        ],
        [
          2326,
          158
        ]
      ]
    }
  ],
  "error_code": 1
}
{
  "filename": "2_66.png",
  "rec_texts_with_position": [
    {
      "te